In [1]:
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import font_manager, rc
import seaborn as sns
import numpy as np

from konlpy.tag import Okt
from sklearn.preprocessing import OneHotEncoder, LabelBinarizer
from sklearn.preprocessing import LabelEncoder

from sklearn.model_selection import train_test_split

from sklearn.feature_extraction.text import TfidfVectorizer

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

# matplotlib의 한글문제를 해결
font_name = font_manager.FontProperties(fname="c:/Windows/Fonts/malgun.ttf").get_name()
# font_name
rc('font', family=font_name)

In [ ]:
# preprocess_text 함수 적용된 데이터 (시간이 오래걸려 한번 수행 후 csv파일로 저장해둠.)
df = pd.read_csv('dataset/Pre-processing_news.csv')
df

,PressCompany,Category,Document
0,뉴스1,정치,독도 일본 땅 주장 역사 국제 법 고유 영토 상보 정부 강력히 항의 한일 관계 도움...
1,파이낸셜뉴스,정치,김정은 요새 순항미사일 쏴 대는 까닭 주일 사이 차례 순항미사일 도발 반길주 유엔 ...
2,오마이뉴스,정치,듣도 보도 못 조선 핼로윈 특별법 이태원 참사 특별법 핼로윈 특별법 이라 칭해 정식...
3,연합뉴스,정치,이재명 오늘 신년 회견 총선 각오 밝히고 민주당 지지 호소 정권 비판 하며 대안 제...
4,국제신문,정치,속보 대통령 이태원 특별법 거부권 행사 취임 후 윤석열 대통령 이태원 특별법 이태원...
...,...,...,...
6967,블로터,IT/과학,흑자 달성 디스플레이 과제 재무 건전 디스플레이 파주 공장 전경 사진 디스플레이 예...
6968,머니투데이,IT/과학,하반신 마비 쥐 신약 맞고 걸었다 영상 다리 근육 뻣뻣해지다가 마비 되는 질환 생명...
6969,블로터,IT/과학,경영 불안 요기 이정환 대표 사임 설 마케팅 확대 제동 불가피 이정환 요기 대표 사...
6970,디지털타임스,IT/과학,삼바 영업 익 첫 돌파 바이오 연대기 획 국내 제약 바이오 기업 중 최초 화이자 위...


### 변수 생성 (Word2Vec(본문), One-Hot(발행사), Label(카테고리))

In [3]:
import gensim
from gensim.models import Word2Vec
from nltk.tokenize import word_tokenize

# 데이터프레임에서 'Document' 열을 가져와서 텍스트 데이터를 리스트로 변환
documents = df['Document'].tolist()

# 각 문서를 토큰화하여 리스트로 변환
tokenized_documents = [word_tokenize(document) for document in documents]

# Word2Vec 모델 생성
model = Word2Vec(sentences=tokenized_documents, vector_size=100, window=5, min_count=1, sg=0)

# 모델 학습
model.train(tokenized_documents, total_examples=len(tokenized_documents), epochs=10)

# 각 문서의 단어 벡터를 평균하여 고정 크기의 벡터로 변환
document_vectors_np = np.array([np.mean(model.wv[words], axis=0) for words in tokenized_documents])

In [4]:
# One-Hot Encoding : 발행사 
# OneHotEncoder 객체 생성
ohe = OneHotEncoder(categories='auto', handle_unknown='ignore') 

# 카테고리 열을 2차원 배열로 변환하여 원-핫 인코딩
press_ohe = ohe.fit_transform(df['PressCompany'].values.reshape(-1, 1))

In [5]:
# Label : 카테고리 (target data)
label_encoder = LabelEncoder()
df['CategoryEncoded'] = label_encoder.fit_transform(df['Category'])

In [66]:
# 각 범주와 매핑된 정수 값 확인
category_mappings = {index: label for index, label in enumerate(label_encoder.classes_)}
print(category_mappings)

{0: 'IT/과학', 1: '경제', 2: '사회', 3: '생활/문화', 4: '정치'}


### 함수 생성

In [6]:
def split_data(document, press):
    # 학습용 데이터와 테스트용 데이터로 나누기
    x_train, x_test, y_train, y_test = train_test_split(
        np.concatenate((document, press), axis=1),  # 두 특성을 결합하여 x로 사용
        np.array(df['CategoryEncoded']),  # 타겟 데이터
        test_size=0.3,  # 테스트 데이터의 비율 설정
        random_state=23  # 랜덤 시드 설정 (재현성을 위해)
    )

    # 분리된 데이터 확인
    print("학습 세트 크기:", x_train.shape, y_train.shape)
    print("테스트 세트 크기:", x_test.shape, y_test.shape)
    
    return x_train, x_test, y_train, y_test

In [7]:
def fit_model(model, x_train, x_test, y_train, y_test):
    # 모델 학습
    model.fit(x_train, y_train)
    
    # 테스트 데이터에 대한 예측
    y_pred = model.predict(x_test)

    # 정확도 스코어 계산
    accuracy = accuracy_score(y_test, y_pred)
    print("정확도:", accuracy)

    # 분류 보고서 출력
    report = classification_report(y_test, y_pred)
    print("분류 보고서:\n", report)

### SVM (Word2Vec(본문), One-Hot(발행사), Label(카테고리)) 적용)

In [9]:
from sklearn.svm import SVC

x_train, x_test, y_train, y_test = split_data(document_vectors_np, press_ohe.toarray())

# SVM 분류기 생성
svm_classifier = SVC(C=10, gamma=0.1, kernel='rbf')

fit_model(svm_classifier, x_train, x_test, y_train, y_test)

학습 세트 크기: (4880, 182) (4880,)
테스트 세트 크기: (2092, 182) (2092,)
정확도: 0.8833652007648184
분류 보고서:
               precision    recall  f1-score   support

           0       0.87      0.89      0.88       442
           1       0.86      0.86      0.86       414
           2       0.85      0.86      0.85       404
           3       0.88      0.85      0.86       426
           4       0.96      0.97      0.96       406

    accuracy                           0.88      2092
   macro avg       0.88      0.88      0.88      2092
weighted avg       0.88      0.88      0.88      2092



### 입력데이터 카테고리 예측하기
- https://n.news.naver.com/mnews/article/029/0002853449 -> 입력데이터 URL

In [ ]:
# https://www.ranks.nl/stopwords/korean
# 텍스트 파일로 저장한 한국어 불용어 사전을 읽어오기
with open('dataset/Korean Stopwords.txt', 'r', encoding='utf-8') as file:
    stopwords = [line.strip() for line in file]

# 읽어온 불용어 리스트 확인
print(stopwords[:50])

['아', '휴', '아이구', '아이쿠', '아이고', '어', '나', '우리', '저희', '따라', '의해', '을', '를', '에', '의', '가', '으로', '로', '에게', '뿐이다', '의거하여', '근거하여', '입각하여', '기준으로', '예하면', '예를 들면', '예를 들자면', '저', '소인', '소생', '저희', '지말고', '하지마', '하지마라', '다른', '물론', '또한', '그리고', '비길수 없다', '해서는 안된다', '뿐만 아니라', '만이 아니다', '만은 아니다', '막론하고', '관계없이', '그치지 않다', '그러나', '그런데', '하지만', '든간에']


In [16]:
tokenizer = Okt()  # 형태소 분석기 초기화

# 텍스트 데이터를 전처리 (명사, 형용사, 동사만 추출 & 불용어 제거)
def preprocess_text(text):
    tokens = tokenizer.pos(text)  # 텍스트를 형태소로 분리
    filtered_pos_tokens = [word for word, pos in tokens if pos in ['Noun', 'Adjective', 'Verb'] and word not in stopwords] # 명사, 형용사, 동사만 추출 & 불용어 제거
    return " ".join(filtered_pos_tokens)  # 추출된 단어들을 문자열로 변환하여 반환

In [73]:
# 본문 데이터 입력받기
text = input("데이터를 입력하세요. :")

# 텍스트를 토큰화하고, 모델 사전에 있는 단어만 필터링
tokens = word_tokenize(preprocess_text(text))
tokens_in_model = [word for word in tokens if word in model.wv.key_to_index]

# 모델 사전에 있는 단어들의 벡터를 구하여 평균 계산
if tokens_in_model:  # 사전에 있는 단어가 하나라도 있으면
    text_word2vec = np.mean([model.wv[word] for word in tokens_in_model], axis=0)
else:  # 사전에 있는 단어가 하나도 없으면
    text_word2vec = np.zeros(model.vector_size)  # 모델의 벡터 크기에 맞는 0 벡터 생성

# 발행사 입력받기
press = input("\n\n발행사를 입력하세요. :")
ohe_press = ohe.transform(np.array([[press]]))

# SVM으로 카테고리 예측
textInput = np.concatenate((text_word2vec.reshape(1, -1), ohe_press.toarray()), axis=1)
predict = svm_classifier.predict(textInput)
print(f'\n예측된 카테고리는 "{category_mappings[predict[0]]}" 입니다.')

데이터를 입력하세요. :'AI(인공지능)로 기세 탄 삼성전자, 역성장은 멈췄지만 주춤하는 애플.'  스마트폰 시장 경쟁이 AI를 중심으로 치열해지고 있는 가운데 삼성전자가 올해 자사 첫 AI폰 '갤럭시S24'를 내놓고 앞서서 달리기 시작했다. 폴더블폰에 이어 AI폰도 먼저 내놓으면서 혁신에서 애플보다 앞서가고 있다. 삼성전자가 갤S24로 반등을 꾀하는 가운데 애플은 AI 경쟁에서 뒤에 서고 중국 시장에서 타격을 입어 올해 보릿고개가 예상된다.  ◇ 애플, 매출 선방에도 '아이폰' 성장 먹구름  애플은 지난 1일(현지시간) 전년 대비 2% 가량 증가한 지난해 4분기 실적을 내놨다. 지난해 4분기 매출은 1195억8000만 달러(약 159조2805억원)로, 2022년 4분기부터 이어진 역성장을 마감했다. 지난해 9월 출시한 '아이폰15' 시리즈의 선방 덕분이다. 같은 해 4분기 아이폰 매출은 시장 예상치인 686억 달러(약 92조원)를 넘어선 697억 달러(약 93조원)를 기록했다.  비록 역성장을 멈추긴 했지만 애플의 미래는 밝지 않다. '중국 리스크'와 함께 삼성전자의 폴더블폰·AI폰 기세, 규제 이슈 등이 먹구름을 드리우고 있다. 실제 애플의 성적은 같은 기간 MS와 아마존, 구글 모회사인 알파벳의 매출이 각각 17%, 14%, 13%로 성장한 것에 비해 초라하다.  스마트폰 시장이 포화하고 애플워치·아이패드 인기가 시들해진 데다 생성형 AI의 등장으로 새 기기와 기술 등이 등장하면서 견고하던 애플 생태계도 금이 가기 시작했다. 특히 중국 시장의 부진이 뼈아프다. 4분기에 대부분 지역에서 아이폰 매출이 증가했지만, 세 번째로 큰 시장인 중국 매출이 1년 전보다 13% 줄었다. 중국 시장에서 4분기 아이폰 매출은 월가의 예상치인 235억 달러(약 31조4000억원)보다 저조한 208억 달러(약 27조8000억원)에 그쳤다. 중국의 견제와 '메이트60'을 필두로 한 화웨이의 부활 때문이다.  이에 더해 EU(유럽연합)가 오는 3월부터 거대 기업의 폐쇄적인 플랫폼